![](img/logo_ucm.jpg)

# Experiments

En MLflow, un _experiment_ es una entidad que agrupa todos los ensayos de un proyecto de machine learning bajo un mismo nombre o ID. Un experimento permite organizar, comparar y gestionar los diferentes intentos de entrenamiento de modelos, pruebas y ajustes que se realicen. 

Cada experimento puede contener múltiples ejecuciones (_runs_) individuales que registran detalles específicos como parámetros, métricas, artefactos y versiones de código.

`````{admonition} runs comparision
:class: important
Dentro de cada _experiment_, se permite realizar comparaciones entre _runs_, lo que permite evaluar qué modelo funciona mejor.
`````

## Primer experimento

Para seguir un guión de buenas prácticas, cada proyecto de ML que tengamos que afrontar sea encapsulado dentro de un experimento. Recordemos que todo proyecto de ML suele tener un ciclo de vida con diversas fases. Los _experiments_ nos pueden ayudar, principalmente, en la fase de modelización (_modeling_). 


```{figure} img/ml_cycle_handwritten.png
---
name: ml lifecycle
align: center
---
ML Lifecycle.
```


### Modelo simple

Para cuando llegamos a la modelización, habremos trabajado previamente las fases de _Exploratory Data Analysis_ y de _Feature Engineering_. Supongamos partir del dataset `iris`, que es un dataset que no necesita de dichas fases. 

`````{note} Interconexión en el *ML Lifecycle*
:class: important
Aunque en este ejemplo vamos a omitir tanto la fase de Feature Engineering como la de Modeling, a lo largo del curso veremos casos en los que ambas están fuertemente interconectadas.

Por ejemplo, imagina que estás construyendo un pipeline de búsqueda de hiperparámetros óptimos. Es posible que durante esa búsqueda también estés generando nuevas variables, como la media o la mediana de una columna. En ese caso, la generación de la feature no puede separarse del proceso de modelado, ya que forma parte de la estrategia global para optimizar el rendimiento y mejorar el comportamiento. Por ello, esta transformación debe integrarse dentro del pipeline completo y ejecutarse como parte del proceso de modelado.
`````

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

A partir de este punto, para configurar el experimento en MLflow de forma programática, se utiliza el método `mlflow.set_experiment`. Esto creará un experimento si no existe o asociará tu ejecución actual con un experimento. Utilizamos `mlflow.sklearn.autolog()` para habilitar las capacidades automatizadas de MLflow para capturar las métricas de nuestro experimento (concretamente, cuando trabajamos con algoritmos de la librería `scikit-learn`).

In [3]:
import mlflow
from sklearn.metrics import accuracy_score

mlflow.set_experiment("Iris_Predictions")
mlflow.sklearn.autolog()

Como explicamos con anterioridad, para ejecutar tu experimento, tendrás que encerrarlo en una ejecución utilizando la palabra clave `with`. La función `mlflow.start_run` se usa para encargarse de registrar tu ejecución con un nombre específico (_run_name_) para que pueda ser identificado, y guardar el modelo ajustado, junto con el código de evaluación utilizado para calcular las métricas de rendimiento del experimento.

In [6]:
from sklearn.tree import DecisionTreeClassifier

with mlflow.start_run(run_name='classification_model_baseline') as run:
    
    model = DecisionTreeClassifier(random_state=33)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    mlflow.log_metric(key="acc", value=acc)

A partir de esta primera ejecución (_run_), ya observamos como podemos crear un nuevo proyecto dentro de la Interfaz de Usuario (UI) y que, en él, los _runs_ e pueden encolar con el metadatado que deseemos.Asimismo, a la izquierda observamos el Experiment catalogado como _Iris_Predictions_ y la lista de ejecuciones realizadas, 2 OK y 1 KO al haber ejecutado la celta faltando una de las variables, por tanto, MLflow es capaz de registrar errores en los experimentos para su posterior revisión. 


En este primer ejemplo, hemos registrado de forma adicional una métrica. 

```{figure} img/mlflow/experiments1.png
---
name: experiments1
align: center
---
Experiments.
```


Para ello, nos hemos apoyado en el comando `log_metric` con la dupla de clave valor. Si exploramos en la ejecución concreta que acabamos de crear, observaremos un trato especial a esta métrica. Además, tendremos un conjunto de métricas adicionales que son registradas gracias al comando `mlflow.sklearn.autolog()` que, si nos interesase, podríamos deshabilitar. De forma adicional, MLflow registra siempre todos los parámetros con los que hemos completado esta ejecución, también porque tenemos activado el `mlflow.sklearn.autolog()`. 

```{figure} img/mlflow/experiments2.png
---
name: extra metrics
align: center
---
Extra Metric.
```


### Modelo parametrizado

Si probásemos ahora con otra parametrización, veríamos que los nuevos parámetros también se registran en cada momento. 

In [5]:
from sklearn.tree import DecisionTreeClassifier

for i in range(1,10):
    
    with mlflow.start_run(run_name='classification_model_baseline_1') as run:
    
        model = DecisionTreeClassifier(
            random_state=33, 
            max_depth=i, 
        )
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        mlflow.log_metric(key="acc", value=acc)


`````{admonition} grid search
:class: info
Gracias a este registro automático de parámetros, podremos explorar el mejor conjunto (teniendo cuidado de no generar _data leakage_ ni _overfitting_). Es lo que se llamaría el Tuning del modelo o la Hiperparametrización de los modelos.
`````

### Comparación de modelos

Seleccionando todos los experimentos creados hasta ahora, podemos generar una comparativa gráfica para ver cómo evoluciona la métrica en función de cada parámetro. En este caso, a mayor valor de _max_depth_, parece que la métrica de Acc disminuye.

```{figure} img/mlflow/experiments6.png
---
name: experiments6
align: center
---
Graphic Metric vs Parameter comparison.
```

`````{admonition} ¿Por qué puede disminuir la precisión (*accuracy*) al aumentar *max_depth*?
:class: info
Sobreajuste (overfitting). Un árbol muy profundo puede aprender ruido y patrones irrelevantes del conjunto de entrenamiento haciendo que el rendimiento en los datos de validación/prueba baje, aunque el accuracy en entrenamiento sea alto. En definitiva, es un modelo que memoriza en lugar de generalizar.
`````

### Modelo con CV

La validación cruzada (_cross validation_, CV) se utiliza para evaluar la robustez de un modelo ML. Ya tratamos esta técnica en el anexo de {doc}`chapter-1-1-1-data-leakage` pero volvemos sobre ella dada su importancia. Se trata de dividir los datos en múltiples subconjuntos y entrenando y evaluando el modelo varias veces, lo que ayuda a simular la dispersión de la métrica estudiada con diversos _train-test splits_. Esto permite estimar de manera más precisa la capacidad de generalización del modelo a datos no vistos.

El siguiente código mostraría cómo hacerlo. Simplemente con el comando `cross_val_score` podemos orquestar la validación cruzada, en este caso para 5 splits. Sin embargo, recordemos que la validación cruzada realiza, en este caso, 5 ajustes del modelo, lo que implica que obtendrá 5 resultados distintos. En MLflow, no podemos guardar todos ellos, tampoco tiene sentido. Si lo que queremos es explorar la calidad y la robustez, bastaría con monitorizar la media y la desviación típica (o la varianza), respectivamente de las particiones, por ejemplo: 

```python
[0.81, 0.83, 0.78, 0.82, 0.80]  # accuracy en cada split
```

>

Media (mean) -> promedio del rendimiento: 0.808

Desviación típica (std) -> variabilidad de esos resultados ≈ 0.017

In [8]:
from sklearn.model_selection import cross_val_score, KFold

with mlflow.start_run(run_name='classification_model_baseline') as run:
    
    model = DecisionTreeClassifier(random_state=33)
    results = cross_val_score(model, X, y, cv=5)
        
    mlflow.log_metric(key="acc_avg", value=results.mean())
    mlflow.log_metric(key="acc_std", value=results.std())

Si exploramos ahora la UI, observaremos que el modelo es registrado correctamente. Pero los parámetros parecen perderse. No importa, lo que ocurre es que estamos utilizando el sistema de `autolog` y este sistema no es capaz de monitorizar siempre todo lo que desearíamos.


```{figure} img/mlflow/experiments3.png
---
name: experiments3
align: center
---
CV mean metrics.
```

Sin embargo, con MLflow podemos monitorizar los parámetros que deseemos mediante el comando `log_params`. Así, de la misma forma que loggeamos métricas, podemos loggear paramétros. 

In [9]:
with mlflow.start_run(run_name='classification_model_baseline') as run:
    
    model = DecisionTreeClassifier(random_state=33)
    params = model.get_params()
    results = cross_val_score(model, X, y, cv=5)
    
    mlflow.log_metric(key="acc_avg", value=results.mean())
    mlflow.log_metric(key="acc_std", value=results.std())
    mlflow.log_params(params=params)

```{figure} img/mlflow/experiments4.png
---
name: experiments4
align: center
---
CV mean metrics & Params.
```

## Flavors

Dentro de un mismo trabajo de experimentación, podemos decidir cambiar los parámetros y explorar los resultados directamente sobre la UI. Además, como científicos de datos, debemos plantearnos evaluar otros algoritmos/modelos. Podemos probar, por ejemplo XGBoost que utiliza la librería `xgboost`. Como dicha librería tiene un _flavor_ en MLflow (que veremos más adelante), podemos utilizar el propio `autolog` de su librería. 

In [7]:
import xgboost as xgb

mlflow.set_experiment("Iris_Predictions")
mlflow.xgboost.autolog()
with mlflow.start_run(run_name='xgboost_model_baseline') as run:
    
    model = xgb.XGBClassifier(random_state=33) 
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    mlflow.log_metric(key="acc", value=acc)

2025/04/19 13:43:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/johndoe/Documents/software/mlops-course/mlops_course_env/lib/python3.10/site-packages/xgboost/core.py:158: UserWarning: [13:43:33] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats."



```{figure} img/mlflow/experiments5.png
---
name: logo
align: center
---
Figura 6: Iris Experiment, XGBoost run.
```


Los _flavors_ en MLflow son formas estándar de empaquetar modelos de ML para garantizar que puedan ser fácilmente utilizados en diferentes herramientas y entornos. Cada _flavor_ especifica cómo guardar, cargar y usar un modelo en un formato particular dentro de MLflow.

`````{admonition} Listado de _flavors_ oficiales de MLflow
:class: info
https://mlflow.org/docs/latest/models.html#built-in-model-flavors

Los flavors son muy útiles pues nos permiten utilizar una misma interfaz para comparar modelos de distintas familias, homogeneizando la manera de trabajar del equipo. 
`````

`````{admonition} Pyfunc
:class: warning
Aquellos algoritmos que no estén dentro de los _flavors_ de MLflow, tendrán un _flavor_ por defecto llamado _pyfunc_. 
`````

Finalmente, podemos también comparar métricas de las ejecuciones que nos interesen entre **_flavors_**. En la pestaña de _metrics_ podemos ver gráficos comparativos por defecto y construir los nuestros propios. Esto es lo realmente útil puesto que bajo un mismo Experiment podremos evaluar todas las posibles opciones registradas. 

```{figure} img/mlflow/experiments7.png
---
name: experiments7
align: center
---
Comparación entre modelos
```


En la siguiente sección, vamos a revisar algunas de las métricas más importantes sobre las que queremos realizar tracking dependientes del problema/caso de uso a resolver. 